# SideKick — PM Intelligence Agent
**Autonomous agentic assistant for Senior Project Managers**

Capabilities:
- Risk & Blocker Analysis
- Project Status Summarization

Stack: LangGraph · Claude 3.5 Sonnet (OpenRouter) · Gradio

## Cell 1 — Install Dependencies

In [13]:
# Install all required packages via uv
import subprocess

packages = [
    "langgraph",
    "langchain",
    "langchain-openai",
    "gradio",
    "python-dotenv",
    "openai",
]

for pkg in packages:
    result = subprocess.run(
        ["uv", "add", pkg],
        capture_output=True, text=True
    )
    status = "OK" if result.returncode == 0 else "WARN"
    print(f"[{status}] {pkg}")

[OK] langgraph
[OK] langchain
[OK] langchain-openai
[OK] gradio
[OK] python-dotenv
[OK] openai


## Cell 2 — Environment Setup

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()  # Reads from .env file in project root

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL = "anthropic/claude-3.5-sonnet"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if not OPENROUTER_API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not found. "
        "Create a .env file with: OPENROUTER_API_KEY=your_key_here"
    )

print(f"[OK] Model  : {MODEL}")
print(f"[OK] API Key: {'*' * 8}{OPENROUTER_API_KEY[-4:]}")

[OK] Model  : anthropic/claude-3.5-sonnet
[OK] API Key: ********cfa2


## Cell 3 — LLM Client

In [15]:
from langchain_openai import ChatOpenAI

def get_llm(temperature: float = 0.3) -> ChatOpenAI:
    """Return a configured LLM client pointing at OpenRouter."""
    return ChatOpenAI(
        model=MODEL,
        openai_api_key=OPENROUTER_API_KEY,
        openai_api_base=OPENROUTER_BASE_URL,
        temperature=temperature,
        default_headers={
            "HTTP-Referer": "https://sidekick-pm-agent",
            "X-Title": "SideKick PM Agent",
        },
    )

# Smoke test
llm = get_llm()
test_response = llm.invoke("Reply with exactly: SIDEKICK ONLINE")
print(f"[OK] LLM test: {test_response.content.strip()}")

[OK] LLM test: SIDEKICK ONLINE


## Cell 4 — Agent State Schema

In [16]:
from typing import TypedDict, Literal, List

class AgentState(TypedDict):
    """Shared state flowing through the LangGraph graph."""
    user_input: str                        # Raw input from the user
    route: Literal["risk", "status", "both"]  # Determined by supervisor
    risk_output: str                       # Output from Risk/Blocker node
    status_output: str                     # Output from Status Summarizer node
    final_response: str                    # Merged final answer
    chat_history: List[dict]               # Conversation context

print("[OK] AgentState schema defined.")

[OK] AgentState schema defined.


## Cell 5 — Supervisor Node (Router)

In [17]:
from langchain_core.messages import SystemMessage, HumanMessage

SUPERVISOR_SYSTEM = """
You are a routing supervisor for a PM AI assistant.
Given the user's input, decide which specialist to invoke:
  - "risk"   → focus is on risks, blockers, issues, dependencies, or escalations
  - "status" → focus is on project progress, milestones, timelines, or team updates
  - "both"   → the input clearly covers both concerns

Reply with ONLY one of: risk | status | both
No explanation. No punctuation. Just the routing word.
"""

def supervisor_node(state: AgentState) -> AgentState:
    """Classify the user input and set the routing decision."""
    llm = get_llm(temperature=0.0)
    messages = [
        SystemMessage(content=SUPERVISOR_SYSTEM),
        HumanMessage(content=state["user_input"]),
    ]
    result = llm.invoke(messages)
    route = result.content.strip().lower()

    # Fallback safety — default to both if unexpected output
    if route not in ("risk", "status", "both"):
        route = "both"

    return {**state, "route": route}

print("[OK] Supervisor node defined.")

[OK] Supervisor node defined.


## Cell 6 — Risk & Blocker Analyst Node

In [18]:
RISK_SYSTEM = """
You are a Risk & Blocker Analyst for a Senior Project Manager at a software development company.
The PM reports directly to the CEO and works with both technical and non-technical stakeholders.

Your job:
1. Identify all risks, blockers, and dependencies mentioned or implied in the input.
2. Rate each risk: LOW / MEDIUM / HIGH / CRITICAL.
3. Suggest a concrete mitigation or escalation action for each.
4. Flag anything that needs immediate CEO-level visibility.

Output format (Markdown):
## Risk & Blocker Analysis

### Identified Risks
| # | Risk / Blocker | Severity | Mitigation Action |
|---|---------------|----------|-------------------|
...

### CEO Escalation Items
- (list any items needing executive visibility, or "None at this time")

### Recommended Next Steps
- (3–5 clear, actionable PM steps)

Be concise, direct, and professional. No filler.
"""

def risk_node(state: AgentState) -> AgentState:
    """Analyse risks and blockers from the PM's input."""
    if state["route"] not in ("risk", "both"):
        return {**state, "risk_output": ""}

    llm = get_llm(temperature=0.2)
    messages = [
        SystemMessage(content=RISK_SYSTEM),
        HumanMessage(content=state["user_input"]),
    ]
    result = llm.invoke(messages)
    return {**state, "risk_output": result.content.strip()}

print("[OK] Risk & Blocker node defined.")

[OK] Risk & Blocker node defined.


## Cell 7 — Project Status Summarizer Node

In [19]:
STATUS_SYSTEM = """
You are a Project Status Summarizer for a Senior Project Manager at a software development company.
The PM reports directly to the CEO and regularly communicates with both technical and non-technical stakeholders.

Your job:
1. Produce a clear, executive-ready project status summary from the input provided.
2. Highlight overall health: ON TRACK / AT RISK / OFF TRACK.
3. Summarize progress against key milestones or goals.
4. Identify team or delivery dependencies.
5. Note what's been completed, what's in progress, and what's upcoming.

Output format (Markdown):
## Project Status Summary

**Overall Health:** ON TRACK | AT RISK | OFF TRACK

### Progress Snapshot
- Completed: ...
- In Progress: ...
- Upcoming: ...

### Milestone Status
| Milestone | Status | Notes |
|-----------|--------|-------|
...

### Key Dependencies
- ...

### Executive Summary (2–3 sentences)
...

Be concise, factual, and CEO-ready. No filler.
"""

def status_node(state: AgentState) -> AgentState:
    """Generate a structured project status summary."""
    if state["route"] not in ("status", "both"):
        return {**state, "status_output": ""}

    llm = get_llm(temperature=0.2)
    messages = [
        SystemMessage(content=STATUS_SYSTEM),
        HumanMessage(content=state["user_input"]),
    ]
    result = llm.invoke(messages)
    return {**state, "status_output": result.content.strip()}

print("[OK] Status Summarizer node defined.")

[OK] Status Summarizer node defined.


## Cell 8 — Aggregator Node (Merge Outputs)

In [20]:
def aggregator_node(state: AgentState) -> AgentState:
    """Combine specialist outputs into a single final response."""
    parts = []

    if state.get("status_output"):
        parts.append(state["status_output"])

    if state.get("risk_output"):
        if parts:
            parts.append("---")
        parts.append(state["risk_output"])

    if not parts:
        parts.append("I was unable to produce a useful analysis from that input. Please provide more project context.")

    return {**state, "final_response": "\n\n".join(parts)}

print("[OK] Aggregator node defined.")

[OK] Aggregator node defined.


## Cell 9 — Build the LangGraph

In [21]:
from langgraph.graph import StateGraph, END

def route_after_supervisor(state: AgentState) -> str:
    """Conditional edge: fan out to risk, status, or both nodes."""
    return state["route"]  # "risk" | "status" | "both"


def build_graph() -> StateGraph:
    """Assemble and compile the LangGraph StateGraph."""
    graph = StateGraph(AgentState)

    # Add nodes
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("risk", risk_node)
    graph.add_node("status", status_node)
    graph.add_node("aggregator", aggregator_node)

    # Entry point
    graph.set_entry_point("supervisor")

    # Conditional routing from supervisor
    graph.add_conditional_edges(
        "supervisor",
        route_after_supervisor,
        {
            "risk": "risk",
            "status": "status",
            "both": "risk",  # "both" path: risk → status → aggregator
        },
    )

    # After risk node: if route is "both", continue to status; else aggregate
    graph.add_conditional_edges(
        "risk",
        lambda s: "status" if s["route"] == "both" else "aggregator",
        {
            "status": "status",
            "aggregator": "aggregator",
        },
    )

    # Status always leads to aggregator
    graph.add_edge("status", "aggregator")

    # Aggregator is the terminal node
    graph.add_edge("aggregator", END)

    return graph.compile()


app_graph = build_graph()
print("[OK] LangGraph compiled successfully.")

[OK] LangGraph compiled successfully.


## Cell 10 — Core Agent Runner

In [22]:
def run_sidekick(user_input: str, chat_history: list = None) -> str:
    """
    Run the SideKick agent graph on a user message.
    Returns the final formatted response string.
    """
    if not user_input or not user_input.strip():
        return "Please describe your project situation so I can assist you."

    initial_state: AgentState = {
        "user_input": user_input.strip(),
        "route": "both",          # Will be overwritten by supervisor
        "risk_output": "",
        "status_output": "",
        "final_response": "",
        "chat_history": chat_history or [],
    }

    result = app_graph.invoke(initial_state)
    return result["final_response"]

print("[OK] run_sidekick() runner defined.")

[OK] run_sidekick() runner defined.


## Cell 11 — Unit Tests

In [23]:
import traceback

TESTS = [
    {
        "name": "Risk-only routing",
        "input": "The backend team is blocked on API credentials from the client. "
                 "We've been waiting 5 days and sprint delivery is at risk.",
        "expect_route": "risk",
    },
    {
        "name": "Status-only routing",
        "input": "We completed the UI mockups last week. Dev is 60% done with the "
                 "authentication module. QA kickoff is scheduled for Friday.",
        "expect_route": "status",
    },
    {
        "name": "Both routing",
        "input": "Sprint 4 is 70% complete. The payment gateway integration is at risk "
                 "due to a third-party API change. Team morale is good but velocity dropped.",
        "expect_route": "both",
    },
    {
        "name": "Empty input guard",
        "input": "   ",
        "expect_route": None,  # Should return guard message, not crash
    },
]

passed = 0
for test in TESTS:
    try:
        if test["expect_route"] is None:
            # Test guard clause
            out = run_sidekick(test["input"])
            assert "Please describe" in out, f"Expected guard message, got: {out[:80]}"
        else:
            # Test supervisor routing
            state = {
                "user_input": test["input"],
                "route": "both",
                "risk_output": "",
                "status_output": "",
                "final_response": "",
                "chat_history": [],
            }
            routed = supervisor_node(state)
            assert routed["route"] == test["expect_route"], (
                f"Expected route '{test['expect_route']}', got '{routed['route']}'"
            )

            # Also test full run produces non-empty output
            full_out = run_sidekick(test["input"])
            assert len(full_out) > 100, "Output too short — likely an error"

        print(f"  [PASS] {test['name']}")
        passed += 1
    except Exception as e:
        print(f"  [FAIL] {test['name']}: {e}")
        traceback.print_exc()

print(f"\nResults: {passed}/{len(TESTS)} tests passed.")

  [PASS] Risk-only routing
  [PASS] Status-only routing
  [PASS] Both routing
  [PASS] Empty input guard

Results: 4/4 tests passed.


## Cell 12 — Gradio UI

In [24]:
import gradio as gr

# ── Gradio chat handler ──────────────────────────────────────────────────────

def chat_handler(message: str, history: list) -> str:
    """Handle a single chat turn."""
    return run_sidekick(message, chat_history=history)


# ── Example prompts ──────────────────────────────────────────────────────────

EXAMPLES = [
    "The dev team is blocked on database access credentials. The client IT dept hasn't responded in 3 days. Sprint ends Friday.",
    "We finished the discovery phase last week. Wireframes are approved. Dev starts Monday. Client sign-off meeting is in 2 weeks.",
    "Sprint 3 is 65% complete. We hit a blocker on the payment gateway — Stripe updated their webhook API. QA is idle waiting for this. The CEO wants a status call tomorrow.",
]

# ── Build Gradio interface ───────────────────────────────────────────────────

with gr.Blocks(
    title="SideKick — PM Intelligence Agent",
    theme=gr.themes.Default(primary_hue="slate"),
) as ui:

    gr.Markdown(
        """
# SideKick — PM Intelligence Agent
Your autonomous project management assistant. Describe your project situation and SideKick will deliver
a structured **Risk & Blocker Analysis** and/or **Project Status Summary** — CEO-ready.
"""
    )

    gr.ChatInterface(
        fn=chat_handler,
        chatbot=gr.Chatbot(
            height=500,
            placeholder="Describe your project situation...",
            show_label=False,
        ),
        textbox=gr.Textbox(
            placeholder="e.g. Sprint 4 is delayed because the third-party integration is blocked...",
            label="Your situation",
            lines=3,
        ),
        examples=EXAMPLES,
        submit_btn="Analyse",
    )

    gr.Markdown(
        """---
**Routing logic:** SideKick automatically detects whether your input requires Risk/Blocker analysis,
Status summarization, or both — and routes accordingly."""
    )

print("[OK] Gradio UI configured.")


C:\Users\nelso\AppData\Local\Temp\ipykernel_16060\2561189498.py:35: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(
c:\Users\nelso\Documents\Projects\agents\.venv\Lib\site-packages\gradio\chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(


[OK] Gradio UI configured.


## Cell 13 — Launch App

In [25]:
# Launch the SideKick Gradio app
# share=True generates a public URL (useful for sharing with stakeholders)
ui.launch(share=False, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
